# Energia gazdálkodás optimalizálása Spark használatával

MySql adatbázisban gyűjtünk adatokat rendszerünk fogyasztásáról, a rendelkezésre álló napelemek termeléséről, illetve az energia ár alakulásáról 15 perces felbontásban.

Az árakat a hupx oldalról API-n keresztül szereztük.

A fogyasztás, termelés és ár adatok rendre a **consumer_profile**, **solar_panel_production** és **hupx_energy_price** táblákban kerülnek letárolásra időbélyeggel együt.



In [32]:
spark.stop()

In [34]:
import findspark
import pyspark.sql
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_timestamp, col, to_utc_timestamp, count, max, monotonically_increasing_id, lit, concat, expr, regexp_replace,lpad
 

findspark.init()

spark = SparkSession.builder \
    .appName("CassandraConncection") \
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.5.0")\
    .config("spark.sql.catalog.client","com.datastax.spark.connector.datasource.CassandraCatalog") \
    .config("spark.sql.catalog.client.spark.cassandra.connection.host","127.0.0.1")\
    .getOrCreate()
    #.config("spark.cassandra.connection.port", "9042")\
    


# Manuális ETL
Ez a blokk tesztadatok manuális betöltését szolgálja fáljokból, valamint egyéb kiegészítő adatok szolgáltatását, ha szükséges.

Itt kell elvégezni az adatok transzformálását, (pl: megfelelő típusra való cast-olás).

A következő blokkok futtatása kihagyható, ha nincsenek ilyen igények.

# Kiolvasás

In [69]:

#reading and transforming csv input, only run this block when needed.
consumption = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",";")\
          .load("../ingest/consumption_profile.csv")

production = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",",")\
          .load("../ingest/production_profile.csv")

price = spark.read.format("csv")\
          .option("header","true")\
          .option("delimiter",";")\
          .option("inferSchema","true")\
          .load("../ingest/price.csv")

consumption.show()
production.show()
price.show()






+----------+----------+--------+
|     Dátum|Negyedórák| Group#4|
+----------+----------+--------+
|2022.01.01|      0:00|0.018103|
|2022.01.01|      0:15|0.018038|
|2022.01.01|      0:30|0.018010|
|2022.01.01|      0:45|0.018095|
|2022.01.01|      1:00|0.018187|
|2022.01.01|      1:15|0.018158|
|2022.01.01|      1:30|0.018087|
|2022.01.01|      1:45|0.018103|
|2022.01.01|      2:00|0.018172|
|2022.01.01|      2:15|0.018167|
|2022.01.01|      2:30|0.018129|
|2022.01.01|      2:45|0.018056|
|2022.01.01|      3:00|0.017923|
|2022.01.01|      3:15|0.017764|
|2022.01.01|      3:30|0.017725|
|2022.01.01|      3:45|0.017770|
|2022.01.01|      4:00|0.017912|
|2022.01.01|      4:15|0.018011|
|2022.01.01|      4:30|0.018170|
|2022.01.01|      4:45|0.018460|
+----------+----------+--------+
only showing top 20 rows

+----------------+--------------+
|            time|production_(W)|
+----------------+--------------+
|01.01.2022 00:00|             0|
|01.01.2022 00:15|             0|
|01.01.2022 

In [70]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
production_final = production.withColumn("timestamp",to_timestamp(col("time"),"dd.MM.yyyy HH:mm"))\
                        .withColumn("profile_id", lit(1567917))\
                        .withColumn("production_(W)", col("production_(W)")*0.00025)\
                        .withColumnRenamed("production_(W)", "production_kwh")\
                        .select("profile_id","timestamp","production_kwh")
                        




In [71]:
spark.conf.set("spark.sql.legacy.timeParserPolicy","LEGACY")
cons_final = consumption.withColumn("time", to_timestamp(concat(col("Dátum"), lit(" "), col("Negyedórák")), "yyyy.MM.dd HH:mm"))\
                        .withColumn("utc_time", to_utc_timestamp(col("time"), "Europe/Budapest"))\
                        .withColumn("consumption_kwh", col("Group#4").cast("double"))\
                        .withColumn("rownum", monotonically_increasing_id())

cons_filter = cons_final.groupBy("utc_time").agg(count("*").alias("count")).filter(col("count")>=2)
cons_filtered = cons_final.join(cons_filter,"utc_time","inner").groupBy("utc_time").agg(max("rownum").alias("rownum"))\
                          .withColumn("modified_utc_time", expr("utc_time + INTERVAL 1 HOUR"))\
                          .select("modified_utc_time","rownum")
cons_final = cons_final.join(cons_filtered,"rownum","left_outer")\
                       .withColumn("timestamp", expr("IFNULL(modified_utc_time, utc_time)"))\
                       .withColumn("profile_id", lit(4))\
                       .select("profile_id","timestamp","consumption_kwh")

cons_final.show()

+----------+-------------------+---------------+
|profile_id|          timestamp|consumption_kwh|
+----------+-------------------+---------------+
|         4|2021-12-31 23:00:00|       0.018103|
|         4|2021-12-31 23:15:00|       0.018038|
|         4|2021-12-31 23:30:00|        0.01801|
|         4|2021-12-31 23:45:00|       0.018095|
|         4|2022-01-01 00:00:00|       0.018187|
|         4|2022-01-01 00:15:00|       0.018158|
|         4|2022-01-01 00:30:00|       0.018087|
|         4|2022-01-01 00:45:00|       0.018103|
|         4|2022-01-01 01:00:00|       0.018172|
|         4|2022-01-01 01:15:00|       0.018167|
|         4|2022-01-01 01:30:00|       0.018129|
|         4|2022-01-01 01:45:00|       0.018056|
|         4|2022-01-01 02:00:00|       0.017923|
|         4|2022-01-01 02:15:00|       0.017764|
|         4|2022-01-01 02:30:00|       0.017725|
|         4|2022-01-01 02:45:00|        0.01777|
|         4|2022-01-01 03:00:00|       0.017912|
|         4|2022-01-

In [72]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
price_final = price.withColumn("Hours",regexp_replace("Hours","[HB]",""))\
                   .withColumn("Hours", expr("cast(Hours as int) - 1"))\
                   .withColumn("Quarters",expr("explode(array(':00',':15',':30',':45'))"))\
                   .withColumn("Hours", concat(lpad("Hours",2,"0"),"Quarters"))\
                   .withColumn("time",to_timestamp(concat("Delivery day",lit(" "),"Hours"),"dd.MM.yyyy HH:mm"))\
                   .withColumn("utc_timestamp", to_utc_timestamp("time", "Europe/Budapest"))\
                   .withColumn("rownum", monotonically_increasing_id())

price_filter = price_final.groupBy("utc_timestamp").agg(count("*").alias("count")).filter(col("count")>=2)
price_filtered = price_final.join(price_filter,"utc_timestamp","inner").groupBy("utc_timestamp").agg(max("rownum").alias("rownum"))\
                            .withColumn("modified_utc_timestamp", expr("utc_timestamp + INTERVAL 1 HOUR")).select("modified_utc_timestamp","rownum")
price_final = price_final.join(price_filtered,"rownum","left_outer")\
                         .withColumn("timestamp", expr("IFNULL(modified_utc_timestamp, utc_timestamp)"))\
                         .withColumnRenamed("Prices (EUR/Mwh)","price_eur_per_mwh")\
                         .select("timestamp","price_eur_per_mwh")

                   

price_final.show()
price_final.printSchema()
price_final.count()

+-------------------+-----------------+
|          timestamp|price_eur_per_mwh|
+-------------------+-----------------+
|2021-12-31 23:00:00|            61.84|
|2021-12-31 23:15:00|            61.84|
|2021-12-31 23:30:00|            61.84|
|2021-12-31 23:45:00|            61.84|
|2022-01-01 00:00:00|            41.33|
|2022-01-01 00:15:00|            41.33|
|2022-01-01 00:30:00|            41.33|
|2022-01-01 00:45:00|            41.33|
|2022-01-01 01:00:00|            43.22|
|2022-01-01 01:15:00|            43.22|
|2022-01-01 01:30:00|            43.22|
|2022-01-01 01:45:00|            43.22|
|2022-01-01 02:00:00|            45.46|
|2022-01-01 02:15:00|            45.46|
|2022-01-01 02:30:00|            45.46|
|2022-01-01 02:45:00|            45.46|
|2022-01-01 03:00:00|            37.67|
|2022-01-01 03:15:00|            37.67|
|2022-01-01 03:30:00|            37.67|
|2022-01-01 03:45:00|            37.67|
+-------------------+-----------------+
only showing top 20 rows

root
 |-- time

70080

In [42]:
price_final.filter(col("timestamp")>= "2023-10-29").show()

+-------------------+-----------------+
|          timestamp|price_eur_per_mwh|
+-------------------+-----------------+
|2023-10-29 00:00:00|            12.16|
|2023-10-29 00:15:00|            12.16|
|2023-10-29 00:30:00|            12.16|
|2023-10-29 00:45:00|            12.16|
|2023-10-29 01:00:00|            10.74|
|2023-10-29 01:15:00|            10.74|
|2023-10-29 01:30:00|            10.74|
|2023-10-29 01:45:00|            10.74|
|2023-10-29 02:00:00|             9.99|
|2023-10-29 02:15:00|             9.99|
|2023-10-29 02:30:00|             9.99|
|2023-10-29 02:45:00|             9.99|
|2023-10-29 03:00:00|            10.37|
|2023-10-29 03:15:00|            10.37|
|2023-10-29 03:30:00|            10.37|
|2023-10-29 03:45:00|            10.37|
|2023-10-29 04:00:00|            10.84|
|2023-10-29 04:15:00|            10.84|
|2023-10-29 04:30:00|            10.84|
|2023-10-29 04:45:00|            10.84|
+-------------------+-----------------+
only showing top 20 rows



In [37]:

cons_final.filter(col("timestamp")>="2023-10-29").show()

+----------+-------------------+---------------+
|profile_id|          timestamp|consumption_kwh|
+----------+-------------------+---------------+
|         4|2023-10-29 00:00:00|       0.013662|
|         4|2023-10-29 00:15:00|       0.013723|
|         4|2023-10-29 00:30:00|        0.01369|
|         4|2023-10-29 00:45:00|       0.013678|
|         4|2023-10-29 01:00:00|       0.013752|
|         4|2023-10-29 01:15:00|       0.013751|
|         4|2023-10-29 01:30:00|       0.013664|
|         4|2023-10-29 01:45:00|       0.013726|
|         4|2023-10-29 02:00:00|       0.013828|
|         4|2023-10-29 02:15:00|       0.013858|
|         4|2023-10-29 02:30:00|       0.013951|
|         4|2023-10-29 02:45:00|        0.01416|
|         4|2023-10-29 03:00:00|       0.014413|
|         4|2023-10-29 03:15:00|        0.01453|
|         4|2023-10-29 03:30:00|        0.01472|
|         4|2023-10-29 03:45:00|       0.014925|
|         4|2023-10-29 04:00:00|       0.015148|
|         4|2023-10-

In [46]:
production_final.filter(col("timestamp")>="2023-10-29").show()

+----------+-------------------+--------------+
|profile_id|          timestamp|production_kwh|
+----------+-------------------+--------------+
|   1567917|2023-10-29 00:00:00|           0.0|
|   1567917|2023-10-29 00:15:00|           0.0|
|   1567917|2023-10-29 00:30:00|           0.0|
|   1567917|2023-10-29 00:45:00|           0.0|
|   1567917|2023-10-29 01:00:00|           0.0|
|   1567917|2023-10-29 01:15:00|           0.0|
|   1567917|2023-10-29 01:30:00|           0.0|
|   1567917|2023-10-29 01:45:00|           0.0|
|   1567917|2023-10-29 02:00:00|           0.0|
|   1567917|2023-10-29 02:15:00|           0.0|
|   1567917|2023-10-29 02:30:00|           0.0|
|   1567917|2023-10-29 02:45:00|           0.0|
|   1567917|2023-10-29 03:00:00|           0.0|
|   1567917|2023-10-29 03:15:00|           0.0|
|   1567917|2023-10-29 03:30:00|           0.0|
|   1567917|2023-10-29 03:45:00|           0.0|
|   1567917|2023-10-29 04:00:00|           0.0|
|   1567917|2023-10-29 04:15:00|        

# Transzformálás

In [30]:
cons_final.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="consumption_profile", keyspace="energycom") \
    .save()

In [9]:
production_final.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="production_profile", keyspace="energycom") \
    .save()

In [10]:
price_final.write \
    .format("org.apache.spark.sql.cassandra") \
    .mode("append") \
    .options(table="hupx_energy_price", keyspace="energycom") \
    .save()

# Reading from Cassandra DB


In [35]:
consumer = spark.read \
    .format("org.apache.spark.sql.cassandra") \
    .options(table="consumptionprofile", keyspace="energycom") \
    .load()

consumer.show()

Py4JJavaError: An error occurred while calling o494.load.
: java.lang.NoSuchMethodError: org.apache.spark.sql.catalyst.analysis.NoSuchTableException: method 'void <init>(java.lang.String)' not found
	at com.datastax.spark.connector.datasource.CassandraCatalog$.tableMissing(CassandraCatalog.scala:462)
	at com.datastax.spark.connector.datasource.CassandraCatalog$.$anonfun$getTableMetaData$2(CassandraCatalog.scala:425)
	at java.base/java.util.Optional.orElseThrow(Optional.java:408)
	at com.datastax.spark.connector.datasource.CassandraCatalog$.getTableMetaData(CassandraCatalog.scala:425)
	at org.apache.spark.sql.cassandra.DefaultSource.getTable(DefaultSource.scala:68)
	at org.apache.spark.sql.cassandra.DefaultSource.inferSchema(DefaultSource.scala:72)
	at org.apache.spark.sql.execution.datasources.v2.DataSourceV2Utils$.getTableFromProvider(DataSourceV2Utils.scala:90)
	at org.apache.spark.sql.execution.datasources.v2.DataSourceV2Utils$.loadV2Source(DataSourceV2Utils.scala:140)
	at org.apach

In [73]:
consumer = cons_final
production = production_final
price = price_final

consumer.show()
production.show()
price.show()

+----------+-------------------+---------------+
|profile_id|          timestamp|consumption_kwh|
+----------+-------------------+---------------+
|         4|2021-12-31 23:00:00|       0.018103|
|         4|2021-12-31 23:15:00|       0.018038|
|         4|2021-12-31 23:30:00|        0.01801|
|         4|2021-12-31 23:45:00|       0.018095|
|         4|2022-01-01 00:00:00|       0.018187|
|         4|2022-01-01 00:15:00|       0.018158|
|         4|2022-01-01 00:30:00|       0.018087|
|         4|2022-01-01 00:45:00|       0.018103|
|         4|2022-01-01 01:00:00|       0.018172|
|         4|2022-01-01 01:15:00|       0.018167|
|         4|2022-01-01 01:30:00|       0.018129|
|         4|2022-01-01 01:45:00|       0.018056|
|         4|2022-01-01 02:00:00|       0.017923|
|         4|2022-01-01 02:15:00|       0.017764|
|         4|2022-01-01 02:30:00|       0.017725|
|         4|2022-01-01 02:45:00|        0.01777|
|         4|2022-01-01 03:00:00|       0.017912|
|         4|2022-01-

# Kiindulási adatok egyesítése

Az optimalizáció előkészítéséhez egyesítjük az adatokat egyetlen Pandas DataFrame-ben.

Előzetes szűrés szükséges a fogyasztó rendszer és a termelő egységre, az adatbázisban több egység termelési illetve fogyasztási adatait is tárolhatjuk. A szűrést azonosítók segítségével tehetjük meg.
Egységesség érdekében float típusú mezővé alakítom a három mutatót a számítási feladatok érdekében.

Itt kerülnek létrehozásra az általánosan szükséges paraméterek is, mint például a rendelkezésre álló akkumulátor kapacitása is.

In [74]:
from pyspark.sql.functions import *
# paramaméterek
capacity = 10 #kwh

# szűrés
consumer   = consumer.filter(col("profile_id") == 4)
production = production.filter(col("profile_id") == 1567917)

# összekapcsolás
process = consumer.select("timestamp","consumption_kwh").join(production.select("timestamp","production_kwh"),"timestamp","inner")\
                  .join(price.select("timestamp","price_eur_per_mwh"),"timestamp","inner")

#rendezés join után és cast-olás
process = process.orderBy("timestamp")\
                 .withColumn("consumption_kwh", process["consumption_kwh"].cast("float"))\
                 .withColumn("production_kwh", process["production_kwh"].cast("float"))\
                 .withColumn("price_eur_per_mwh", process["price_eur_per_mwh"].cast("float"))\
                 .toPandas()
process.head()


,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh
0,2022-01-01 00:00:00,0.018187,0.0,41.330002
1,2022-01-01 00:15:00,0.018158,0.0,41.330002
2,2022-01-01 00:30:00,0.018087,0.0,41.330002
3,2022-01-01 00:45:00,0.018103,0.0,41.330002
4,2022-01-01 01:00:00,0.018172,0.0,43.220001


# Optimalizálási feladat

Most, hogy kész a rendezett adatsorunk, alkalmazhatjuk a különböző optimalizációs stratégiákat a minél eredményesebb energia gazdálkodás érdekében. 

A célfüggvény amit választottunk az a fogyasztási igények kiszolgálása a **költség minimalizálásával**. Annál jobb az eredmény, minél kevesebbe került az energiaszolgáltatás az adott időszakra.

különböző megvalósításokat igyekszünk összehasonlítani, hogy megállapíthassuk, melyik megoldás a legkedvezőbb.

# PV only

In [78]:
pv = process.assign(feeding_grid = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, (x['production_kwh'] - x['consumption_kwh'])*0.25, 0),
                    taking_from_grid   = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, (x['consumption_kwh'] - x['production_kwh'])*0.25, 0),
                    sell_price = lambda x: x["feeding_grid"]*x["price_eur_per_mwh"]*0.00025,
                    buy_price = lambda x: x["taking_from_grid"]*x["price_eur_per_mwh"]*0.00025,
                    net_revenue = lambda x: x["sell_price"] - x["buy_price"])

pv.head()

,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh,feeding_grid,taking_from_grid,sell_price,buy_price,net_revenue
0,2022-01-01 00:00:00,0.018187,0.0,41.330002,0.0,0.004547,0.0,0.000047,-0.000047
1,2022-01-01 00:15:00,0.018158,0.0,41.330002,0.0,0.004539,0.0,0.000047,-0.000047
2,2022-01-01 00:30:00,0.018087,0.0,41.330002,0.0,0.004522,0.0,0.000047,-0.000047
3,2022-01-01 00:45:00,0.018103,0.0,41.330002,0.0,0.004526,0.0,0.000047,-0.000047
4,2022-01-01 01:00:00,0.018172,0.0,43.220001,0.0,0.004543,0.0,0.000049,-0.000049


# Simple Strategy

A következő blokkokban egy próba algoritmust implementáltam, mely nem az adatok viszonya alapján készíti el a költség tervet.

Csupán referenciaként szolgáló megoldás.

**Működés:**

Ha az adott időben a termelés fedezi a fogyasztást, akkor többlet keletkezik, a többletből betáplálunk annyit az akkumulátorba, amennyi belefér, az el nem tárolható mennyiséget pedig kitápláljuk a hálózatra.

Másik esetben, mikor nem fedezi a termelés a fogyasztást, akkor a szükséges energiamennyiséget első sorban az akkumulátorban tárolt energiából pótoljuk, hogyha az sem elég, akkor vételezünk a hálózatról.


In [75]:
import numpy as np
# energia többlet és igény kiszámítása a termelés és fogyasztás különbségéből,
# valamint töltöttséget követő oszlopok bevezetése a negyedóra elejére és végére, alapértelmezett értékadás.
basic_p = process.assign(energy_to_store = lambda x: np.where(x['production_kwh'] - x['consumption_kwh'] > 0, (x['production_kwh'] - x['consumption_kwh'])*0.25, 0),
                       energy_to_get   = lambda x: np.where(x['consumption_kwh'] - x['production_kwh'] > 0, (x['consumption_kwh'] - x['production_kwh'])*0.25, 0),
                       battery_charge_start = float(0.00000),
                       battery_charge_end   = float(0.00000))
basic_p.head()

#from pyspark.sql.functions import when
#basic_p = process.withColumn("energy_to_store", when(col("production_kwh")-col("consumption_kwh")>0,col("production_kwh")-col("consumption_kwh"))\
#                          .otherwise(0))\
#               .withColumn("energy_to_get",when(col("consumption_kwh")-col("production_kwh")>0,col("consumption_kwh")-col("production_kwh"))\
#                          .otherwise(0))\
#               .withColumn("battery_charge_at_start",lit(0.00000))\
#               .withColumn("battery_charge_at_end",lit(0.00000))

# kezdő értéket állíthatunk az akkumulátorunknak a vizsgált időszak elején.
# basic_p.at[0,"battery_charge_start"] = 2

#ha tárolni kell akkor mennyit tudunk eltárolni
if basic_p.at[0,'energy_to_store'] > 0:
        temp = basic_p.at[0,'battery_charge_start'] + basic_p.at[0,'energy_to_store']
        if temp <= capacity:
            basic_p.at[0,'battery_charge_end'] = temp
        else:
            basic_p.at[0,'battery_charge_end'] = capacity
            
#ha be kell szerezni, mennyit tudunk szolgáltatni saját tárunkból
else:
        temp = basic_p.at[0,'battery_charge_start'] - basic_p.at[0,'energy_to_get']
        if temp > 0:
            basic_p.at[0,'battery_charge_end'] = temp
        else:
            basic_p.at[0,'battery_charge_end'] = 0

for i in range(1,int(basic_p.size/8)):
    basic_p.at[i,'battery_charge_start'] = basic_p.at[i-1,'battery_charge_end']
    if basic_p.at[i,'energy_to_store'] > 0:
        temp = basic_p.at[i,'battery_charge_start'] + basic_p.at[i,'energy_to_store']
        if temp <= capacity:
            basic_p.at[i,'battery_charge_end'] = temp
        else:
            basic_p.at[i,'battery_charge_end'] = capacity
    else:
        temp = basic_p.at[i,'battery_charge_start'] - basic_p.at[i,'energy_to_get']
        if temp > 0:
            basic_p.at[i,'battery_charge_end'] = temp
        else:
            basic_p.at[i,'battery_charge_end'] = 0
#basic_p.tail(10)

#az előző adatok alapján megállapítani, hogy mennyit fogyasztunk saját, illetve hálózati energiából, 
# valamint mennyi energiát táplálunk be saját akkumulátorunkba, vagy a hálózatba. 
basic_p = basic_p.assign(taking_from_battery = lambda x: np.where(x['battery_charge_start'] - x['battery_charge_end'] > 0, x['battery_charge_start'] - x['battery_charge_end'], 0),
                       feeding_battery   = lambda x: np.where(x['battery_charge_end'] - x['battery_charge_start'] > 0, x['battery_charge_end'] - x['battery_charge_start'], 0),
                       taking_from_grid  = lambda x: x['energy_to_get']-x['taking_from_battery'],
                       feeding_grid      = lambda x: x['energy_to_store']-x['feeding_battery'],
                       price             = lambda x: (x['taking_from_grid']*x['price_eur_per_mwh']-x['feeding_grid']*x['price_eur_per_mwh'])*0.00025) 

basic_p.head(100)



,timestamp,consumption_kwh,production_kwh,price_eur_per_mwh,energy_to_store,energy_to_get,battery_charge_start,battery_charge_end,taking_from_battery,feeding_battery,taking_from_grid,feeding_grid,price
0,2022-01-01 00:00:00,0.018187,0.0,41.330002,0.0,0.004547,0.000000,0.000000,0.000000,0.0,0.004547,0.0,0.000047
1,2022-01-01 00:15:00,0.018158,0.0,41.330002,0.0,0.004539,0.000000,0.000000,0.000000,0.0,0.004539,0.0,0.000047
2,2022-01-01 00:30:00,0.018087,0.0,41.330002,0.0,0.004522,0.000000,0.000000,0.000000,0.0,0.004522,0.0,0.000047
3,2022-01-01 00:45:00,0.018103,0.0,41.330002,0.0,0.004526,0.000000,0.000000,0.000000,0.0,0.004526,0.0,0.000047
4,2022-01-01 01:00:00,0.018172,0.0,43.220001,0.0,0.004543,0.000000,0.000000,0.000000,0.0,0.004543,0.0,0.000049
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2022-01-01 23:45:00,0.018095,0.0,57.080002,0.0,0.004524,2.314730,2.310206,0.004524,0.0,0.000000,0.0,0.000000
96,2022-01-02 00:00:00,0.018187,0.0,52.590000,0.0,0.004547,2.310206,2.305659,0.004547,0.0,0.000000,0.0,0.000000
97,2022-01-02 00:15:00,0.018158,0.0,52.590000,0.0,0.004539,2.305659,2.301120,0.004539,0.0,0.000000,0.0,0.000000
98,2022-01-02 00:30:00,0.018087,0.0,52.590000,0.0,0.004522,2.301120,2.296598,0.004522,0.0,0.000000,0.0,0.000000


In [77]:
simple_total_generation = basic_p.assign(generation = lambda x: x['production_kwh']*0.25)['generation'].sum()
simple_total_self_consumed = basic_p.assign(self_consumed = lambda x: x['production_kwh']*0.25 -x['feeding_grid'])['self_consumed'].sum()
print("Total generation: ",simple_total_generation, " kw")
print("Total_self_consumed: ", simple_total_self_consumed, " kw")
print(basic_p['consumption_kwh'].sum())

Total generation:  7102.7144  kw
Total_self_consumed:  509.71448887999213  kw
1999.9276
